# Simple Simulation (10 Jobs)

This notebook runs a minimal simulation of 10 jobs scheduled by a specified trained model.

- Set EXPERIMENT, WORKLOAD, and optional EPOCH
- Loads model weights (final or a specific checkpoint)
- Initializes the HPC environment
- Overrides the episode to 10 jobs
- Steps through scheduling with the model and reports a concise summary



In [1]:
# Configuration
# If your checkpoint lives under a specific workload folder, set EXPERIMENT_PATH
# (e.g., "lublin_256_uni/MARL_carbon_co2direct"). This avoids coupling to the
# SWF file chosen for testing (e.g., frontloaded sanity datasets).

EXPERIMENT_PATH = "lublin_256_carbon_all1/MARL_carbon_min_only"  # e.g., "lublin_256_uni/MARL_carbon_min_only" overrides EXPERIMENT

# You can point WORKLOAD to any SWF (sanity/frontloaded) without affecting where
# the checkpoint is loaded from when EXPERIMENT_PATH is set.
WORKLOAD = "./data/lublin_256_frontloaded.swf"
EPOCH = 5  # e.g., 300 for a specific checkpoint, or None for final

BACKFILL = 0   # 0=FCFS, 1=backfill enabled
SEED = 42
DEBUG = False
JOBS_PER_EPISODE = 500
delayMaxJobNum = 5

action2_num= 7 + delayMaxJobNum + 1


In [2]:
# Imports
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from compare import RL_MultiAction  
from HPCSimPickJobs import *
from validate import load_ccscheduler_model

# Reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
def initialize_model(experiment_path, workload, epoch):
    # Load model
    exp_for_load = experiment_path if experiment_path else EXPERIMENT
    model = load_ccscheduler_model(exp_for_load, workload, epoch)
    model_device = next(model.actor_net.parameters()).device if hasattr(model, 'actor_net') else torch.device('cpu')
    print("Model loaded on:", model_device)
    return model

def initialize_environment(workload, backfill, debug, seed, jobs_per_episode):
    # Resolve workload path
    if workload.endswith('.swf'):
        workload_file = workload
    else:
        workload_file = f"./data/{workload}.swf"
    print("Using workload:", workload_file)

    # Initialize environment
    env = HPCEnv(backfill=backfill, debug=debug)
    env.my_init(workload_file)
    env.seed(seed)

    # Override job count for the episode
    env._validation_job_sequence_size = jobs_per_episode

    # Prepare episode with exactly JOBS_PER_EPISODE jobs
    env.reset()
    if hasattr(env, '_validation_job_sequence_size'):
        # Bound within available jobs
        max_jobs_available = len(env.loads.all_jobs) - env.start
        actual_job_count = min(env._validation_job_sequence_size, max_jobs_available)
        env.num_job_in_batch = actual_job_count
        env.last_job_in_batch = env.start + env.num_job_in_batch
        print(f"Episode configured for {env.num_job_in_batch} jobs (requested {JOBS_PER_EPISODE})")

    return env


In [4]:
def run_scheduling_loop(env, model, action2_num, delayMaxJobNum, MAX_QUEUE_SIZE):
    # Run scheduling loop
    o = env.build_observation()
    running_num = 0
    step_count = 0

    total_reward = 0.0
    green_reward = 0.0

    scheduled_jobs = []

    while True:
        step_count += 1
        
        # Build action1 mask from env.pairs (like validate.py)
        mask1 = []
        for i in range(MAX_QUEUE_SIZE):
            if i < len(env.pairs) and env.pairs[i][0] is not None:
                mask1.append(0)
            else:
                mask1.append(1)
        
        # Termination when episode jobs processed
        if (env.next_arriving_job_idx >= env.last_job_in_batch and 
            len(env.job_queue) == 0 and len(env.running_jobs) == 0):
            break
        
        # If no jobs to process, try to fast-forward
        if mask1.count(0) == 0 and len(env.job_queue) == 0:
            if hasattr(env, 'fast_forward_to_next_event') and env.fast_forward_to_next_event():
                o = env.build_observation()
                continue
            else:
                break
        
        # Action2 mask
        mask2 = np.zeros(action2_num, dtype=int)
        if running_num < delayMaxJobNum:
            mask2[running_num + 1:delayMaxJobNum + 1] = 1
        
        # Get actions
        a1, a2 = model.eval_action(o, mask1, mask2)
        
        # Capture scheduled job before step
        selected_job = env.job_queue[a1] if a1 < len(env.job_queue) else None
        
        # Step
        o, r, d, r2, sjf_t, f1_t, running_num, greenRwd = env.step(a1, a2)
        total_reward += r
        green_reward += greenRwd
        
        # Track scheduled job
        if selected_job is not None and hasattr(selected_job, 'scheduled_time') and selected_job.scheduled_time != -1:
            scheduled_jobs.append({
                'job_id': getattr(selected_job, 'job_id', None),
                'submit_time': selected_job.submit_time,
                'scheduled_time': selected_job.scheduled_time,
                'request_time': selected_job.request_time,
                'processors': selected_job.request_number_of_processors,
                'carbon_consideration': getattr(selected_job, 'carbon_consideration', -1),
                'power': getattr(selected_job, 'power', 0)
            })
        
        if d:
            break
        # Additional end check when batch jobs submitted
        if env.next_arriving_job_idx >= env.last_job_in_batch and len(env.job_queue) == 0:
            break

    print(f"Steps: {step_count}, total_reward: {total_reward:.4f}, green_reward: {green_reward:.4f}")
    return scheduled_jobs, total_reward, green_reward


In [5]:

## Load in epoch 35 vs 5
# Example usage:

env = initialize_environment(WORKLOAD, BACKFILL, DEBUG, SEED, JOBS_PER_EPISODE)
model_epoch_5 = initialize_model(EXPERIMENT_PATH, WORKLOAD, 5)
model_epoch_35 = initialize_model(EXPERIMENT_PATH, WORKLOAD, 35)

model_list = [model_epoch_5, model_epoch_35]

for model in model_list:
    scheduled_jobs, total_reward, green_reward = run_scheduling_loop(env, model, action2_num, delayMaxJobNum, MAX_QUEUE_SIZE)
    print(total_reward, green_reward)

Using workload: ./data/lublin_256_frontloaded.swf
Initialize Simple HPC Env
loading workloads from dataset: ./data/lublin_256_frontloaded.swf
Using constant power: 500.0 watts per processor
Max Allocated Processors: 256 ;max node: 256 ;max procs: 256 ;max execution time: 124707
Episode configured for 500 jobs (requested 500)
Loading weights from: lublin_256_carbon_all1/MARL_carbon_min_only/checkpoints/epoch_5/
Model loaded on: cpu
Loading weights from: lublin_256_carbon_all1/MARL_carbon_min_only/checkpoints/epoch_35/
Model loaded on: cpu
Steps: 500, total_reward: -15877.6978, green_reward: 0.0000
-15877.697760061437 0.0
Steps: 17, total_reward: 0.0000, green_reward: 0.0000
0.0 0.0


In [6]:
# Results summary

jobs_df = pd.DataFrame(scheduled_jobs)
print(f"Scheduled jobs: {len(jobs_df)}")
display(jobs_df.head(10))

if not jobs_df.empty:
    jobs_df['wait_time'] = jobs_df['scheduled_time'] - jobs_df['submit_time']
    print("\nSummary:")
    print("Avg wait (s):", jobs_df['wait_time'].mean())
    print("Avg runtime (s):", jobs_df['request_time'].mean())
    print("Avg processors:", jobs_df['processors'].mean())
    print("Avg carbon_consideration:", jobs_df['carbon_consideration'].mean())




Scheduled jobs: 0


""


In [7]:
# Visualization: Allocated processors vs carbon intensity over time

if not jobs_df.empty:
    # Choose time bin size (seconds)
    BIN_SECONDS = 3600  # 1 hour bins; try 900 for 15 minutes

    # Determine plotting window from scheduled jobs
    valid = jobs_df.dropna(subset=['scheduled_time'])
    if not valid.empty:
        start_t = valid['scheduled_time'].min()
        end_t = (valid['scheduled_time'] + valid['request_time']).max()

        # Build bins and accumulator for processors
        time_bins = np.arange(start_t, end_t + BIN_SECONDS, BIN_SECONDS, dtype=int)
        if len(time_bins) < 2:
            time_bins = np.array([start_t, start_t + BIN_SECONDS], dtype=int)
        proc_alloc = np.zeros(len(time_bins) - 1, dtype=float)

        # Accumulate processors per bin for running intervals
        for _, row in valid.iterrows():
            st = int(row['scheduled_time'])
            et = int(row['scheduled_time'] + row['request_time'])
            procs = float(row['processors'])
            # Overlap mask: bin [t_i, t_{i+1}) intersects [st, et)
            left = time_bins[:-1]
            right = time_bins[1:]
            mask = (left < et) & (right > st)
            proc_alloc[mask] += procs

        # Carbon intensity per bin (use intensity at bin start)
        carbon_vals = []
        for t in time_bins[:-1]:
            slot = env.cluster.carbonIntensity.getCarbonIntensitySlot(t)
            carbon_vals.append(slot[0]['carbonIntensity'] if slot else 0.0)
        carbon_vals = np.array(carbon_vals, dtype=float)

        # X coordinates in hours relative to start
        rel_hours = (time_bins[:-1] - start_t) / 3600.0

        # Plot
        fig, ax1 = plt.subplots(figsize=(10, 4))
        ax1.bar(rel_hours, proc_alloc, width=(BIN_SECONDS/3600.0), align='edge',
                color='#1f77b4', alpha=0.6, edgecolor='#1f77b4', label='Allocated processors')
        ax1.set_xlabel('Time (hours from episode start)')
        ax1.set_ylabel('Allocated processors')

        ax2 = ax1.twinx()
        ax2.plot(rel_hours, carbon_vals, color='green', linewidth=2.0, label='Carbon intensity (gCO2/kWh)')
        ax2.set_ylabel('Carbon intensity (gCO2/kWh)')

        # Legends
        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax1.legend(h1 + h2, l1 + l2, loc='upper right')
        plt.title('Allocated processors vs Carbon intensity')
        plt.tight_layout()
        plt.show()
    else:
        print('No scheduled jobs to visualize yet.')
else:
    print('No jobs were scheduled.')


No jobs were scheduled.
